<a href="https://colab.research.google.com/github/ProfessorDong/Deep-Learning-Course-Examples/blob/master/CNN_Examples/CNN_Demo1_Visualizing_Filters_FeatureMaps.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Demo 1: Visualizing CNN Filters and Feature Maps

**ELC5365 Deep Learning** | Baylor University

This demo reveals what a CNN "sees" at each layer. We'll explore:
1. The learned filters of a pretrained VGG-16 (they look like edge detectors!)
2. Feature maps at different depths — from edges to high-level concepts
3. Parameter counts across famous architectures
4. Activation maximization — what pattern maximally excites a neuron?

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.models as models
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

---
## Section 1: What Do CNN Filters Look Like?

The first convolutional layer of a trained CNN learns **low-level features** — edges,
colors, and textures. These filters resemble **Gabor filters** from signal processing.

Let's load VGG-16 and visualize its 64 first-layer filters.

In [ ]:
# Load pretrained VGG-16
vgg16 = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
vgg16.eval()
print("VGG-16 first 6 feature layers:")
for i, layer in enumerate(vgg16.features[:6]):
    print(f"  [{i}] {layer}")

In [ ]:
# Extract the 64 filters from conv1_1 (shape: 64 x 3 x 3 x 3)
filters = vgg16.features[0].weight.data.clone().cpu()
print(f"Conv1_1 filters shape: {filters.shape}")
print(f"  -> 64 filters, each 3x3, across 3 color channels (RGB)")

# Normalize each filter to [0, 1] for display
f_min, f_max = filters.min(), filters.max()
filters_norm = (filters - f_min) / (f_max - f_min)

# Display all 64 filters in an 8x8 grid
fig, axes = plt.subplots(8, 8, figsize=(12, 12))
fig.suptitle('VGG-16 Conv1_1: 64 Learned Filters (3x3 RGB)', fontsize=16, fontweight='bold')
for i, ax in enumerate(axes.flat):
    img = filters_norm[i].permute(1, 2, 0).numpy()
    ax.imshow(img, interpolation='nearest')
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

print("Notice: these filters detect edges at various orientations and color gradients!")
print("They closely resemble Gabor filters used in traditional signal processing.")

---
## Section 2: Feature Maps — Watching the CNN Think

Now let's feed an image through VGG-16 and **visualize the feature maps** at
different depths. You'll see a clear progression:

| Layer | What it captures |
|-------|------------------|
| conv1 | Edges, color gradients |
| conv2 | Textures, simple patterns |
| conv3 | Complex textures, parts |
| conv4 | Object parts |
| conv5 | High-level semantic features |

In [ ]:
# Load a CIFAR-10 test image and resize for VGG-16
cifar10_test = torchvision.datasets.CIFAR10(root='./data', train=False, download=True)
class_names = cifar10_test.classes

# Pick an interesting image -- try different indices!
sample_idx = 3
sample_img, sample_label = cifar10_test[sample_idx]

print(f"Original image: {class_names[sample_label]} (32x32)")
plt.figure(figsize=(3, 3))
plt.imshow(sample_img)
plt.title(f'Input: {class_names[sample_label]}', fontsize=14, fontweight='bold')
plt.axis('off')
plt.show()

# Preprocess for VGG-16: resize to 224x224, normalize with ImageNet stats
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
input_tensor = preprocess(sample_img).unsqueeze(0)
print(f"Preprocessed tensor shape: {input_tensor.shape}")

In [ ]:
# Extract feature maps at key layers using forward hooks
feature_maps = {}
target_layers = {
    'conv1_1 (64)': 0,
    'conv2_1 (128)': 5,
    'conv3_1 (256)': 10,
    'conv4_1 (512)': 17,
    'conv5_1 (512)': 24
}

hooks = []
for name, idx in target_layers.items():
    def hook_fn(module, input, output, name=name):
        feature_maps[name] = output.detach()
    hooks.append(vgg16.features[idx].register_forward_hook(hook_fn))

with torch.no_grad():
    output = vgg16(input_tensor)

for h in hooks:
    h.remove()

print("Feature map sizes at each layer:")
for name, fmap in feature_maps.items():
    print(f"  {name}: {list(fmap.shape)}")

In [ ]:
# Visualize 16 feature maps from each layer
fig, axes = plt.subplots(5, 16, figsize=(24, 8))
fig.suptitle('Feature Maps at Different Depths of VGG-16', fontsize=18, fontweight='bold')

for row, (name, fmap) in enumerate(feature_maps.items()):
    fmap_np = fmap[0].cpu().numpy()
    for col in range(16):
        ax = axes[row, col]
        ax.imshow(fmap_np[col], cmap='viridis')
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(name.split('(')[0], fontsize=9, rotation=0,
                         labelpad=60, va='center')

plt.tight_layout()
plt.show()

print("Key observations:")
print("  conv1: responds to edges and color boundaries")
print("  conv2: captures textures and simple combinations")
print("  conv3: more complex patterns emerge")
print("  conv4-5: abstract, high-level features (object parts)")
print("  Spatial resolution decreases with depth (due to pooling)")

In [ ]:
# Show the STRONGEST activation map at each layer
fig, axes = plt.subplots(1, 6, figsize=(20, 4))
fig.suptitle('Strongest Activation at Each Layer', fontsize=16, fontweight='bold')

axes[0].imshow(sample_img)
axes[0].set_title('Original', fontsize=11)
axes[0].axis('off')

for i, (name, fmap) in enumerate(feature_maps.items()):
    fmap_np = fmap[0].cpu().numpy()
    strongest = fmap_np.max(axis=(1, 2)).argmax()
    axes[i+1].imshow(fmap_np[strongest], cmap='hot')
    axes[i+1].set_title(name.split('(')[0], fontsize=11)
    axes[i+1].axis('off')

plt.tight_layout()
plt.show()

---
## Section 3: Comparing Architecture Parameter Counts

How do the famous CNN architectures compare in size?

**Deeper networks don't always mean more parameters.** ResNet-50 is much deeper
than VGG-16 (50 vs 16 layers) but uses 5x fewer parameters!

In [ ]:
class LeNet5(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 6, 5), nn.Tanh(), nn.AvgPool2d(2),
            nn.Conv2d(6, 16, 5), nn.Tanh(), nn.AvgPool2d(2))
        self.classifier = nn.Sequential(
            nn.Linear(16*5*5, 120), nn.Tanh(),
            nn.Linear(120, 84), nn.Tanh(),
            nn.Linear(84, 10))
    def forward(self, x):
        return self.classifier(self.features(x).view(x.size(0), -1))

architectures = {
    'LeNet-5\n(1998)': LeNet5(),
    'AlexNet\n(2012)': models.alexnet(),
    'VGG-16\n(2014)': models.vgg16(),
    'GoogLeNet\n(2014)': models.googlenet(),
    'ResNet-18\n(2015)': models.resnet18(),
    'ResNet-50\n(2015)': models.resnet50(),
}

param_counts = {}
for name, model in architectures.items():
    count = sum(p.numel() for p in model.parameters())
    param_counts[name] = count
    print(f"{name.replace(chr(10), ' '):20s} {count:>14,} params  ({count/1e6:.1f}M)")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
names = list(param_counts.keys())
counts = [v / 1e6 for v in param_counts.values()]
colors = ['#2ecc71', '#3498db', '#e74c3c', '#f39c12', '#9b59b6', '#1abc9c']

bars = ax.bar(names, counts, color=colors, edgecolor='black', linewidth=0.8)
for bar, c in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
            f'{c:.1f}M', ha='center', va='bottom', fontweight='bold', fontsize=11)

ax.set_ylabel('Parameters (Millions)', fontsize=13)
ax.set_title('CNN Architecture Parameter Comparison', fontsize=16, fontweight='bold')
ax.set_ylim(0, max(counts) * 1.15)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print("VGG-16 has 138M parameters, but most are in the FC layers!")
print("GoogLeNet achieves BETTER accuracy than VGG with ~28x fewer parameters!")

---
## Section 4: Activation Maximization — What Excites a Neuron?

We can ask: **what input image would maximally activate a specific filter?**

Starting from random noise, we use **gradient ascent** on the input to maximize
a filter's activation. The resulting pattern reveals what "concept" the filter detects.

In [ ]:
def activation_maximization(model, layer_idx, filter_idx, size=128, iterations=200, lr=0.1):
    """Generate an image that maximally activates a specific filter via gradient ascent."""
    input_img = torch.randn(1, 3, size, size, requires_grad=True, device=device)
    optimizer = torch.optim.Adam([input_img], lr=lr)
    model = model.to(device).eval()

    activation = {}
    def hook(module, inp, out):
        activation['value'] = out
    handle = model.features[layer_idx].register_forward_hook(hook)

    for _ in range(iterations):
        optimizer.zero_grad()
        model(input_img)
        loss = -activation['value'][0, filter_idx].mean()  # Maximize activation
        loss.backward()
        optimizer.step()

    handle.remove()
    img = input_img.detach().cpu().squeeze()
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    return img.permute(1, 2, 0).numpy()

print("Activation maximization: gradient ascent to find the input that excites a neuron most.")

In [ ]:
# Generate patterns for 6 filters in conv3_1 (layer index 10)
target_layer = 10  # conv3_1 -- deeper layers produce more interesting patterns
filter_indices = [5, 15, 30, 50, 80, 120]

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
fig.suptitle('Activation Maximization: conv3_1 Filters\n'
             '"What pattern makes each neuron fire most strongly?"',
             fontsize=15, fontweight='bold')

for ax, f_idx in zip(axes.flat, filter_indices):
    print(f"  Optimizing filter {f_idx}...")
    pattern = activation_maximization(vgg16, target_layer, f_idx, size=128, iterations=200)
    ax.imshow(pattern)
    ax.set_title(f'Filter #{f_idx}', fontsize=12)
    ax.axis('off')

plt.tight_layout()
plt.show()

print("\nEach pattern shows the texture/structure that maximally excites that filter.")
print("Try changing target_layer to 0 (edges), 17 (parts), or 24 (high-level)!")

---
## Summary

| Technique | What it reveals |
|-----------|----------------|
| **Filter visualization** | The learned edge/color detectors (conv1 weights) |
| **Feature maps** | How each layer transforms the input image |
| **Activation maximization** | The ideal input pattern for each filter |

**Key takeaway:** CNNs build a hierarchy: simple edges -> textures -> parts -> objects.
This hierarchical feature learning is what makes deep learning so powerful for vision.